In [2]:
import pandas as pd
import joblib

In [3]:
credit_score = pd.read_csv('../../creditscore_anon_cw_july2023.csv')
credit_score.head()

,A0,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,1,4,31,Female,5,3,1,3,0,1,...,0,4,0,0,0,0,0,0,0,28553.30
1,2,4,31,Female,5,3,1,3,0,1,...,0,2,0,0,0,0,0,0,0,10410.00
2,3,3,26,Female,7,3,1,0,0,1,...,0,7,0,104,0,0,0,0,0,10979.06
3,4,0,63,Male,7,3,3,2,0,2,...,0,3,0,0,0,0,0,0,0,10242.00
4,5,0,26,Male,5,3,1,1,0,1,...,1,0,0,0,0,0,0,0,200,1489.00


# Data cleaning

The dataset seems to include a column A0 which is the ID or the row that wa can get rid of, it doesnt bring anything

In [4]:
credit_score = credit_score.drop(columns=['A0'], axis=1)

In [5]:
credit_score.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,4,31,Female,5,3,1,3,0,1,24,...,0,4,0,0,0,0,0,0,0,28553.30
1,4,31,Female,5,3,1,3,0,1,24,...,0,2,0,0,0,0,0,0,0,10410.00
2,3,26,Female,7,3,1,0,0,1,24,...,0,7,0,104,0,0,0,0,0,10979.06
3,0,63,Male,7,3,3,2,0,2,3,...,0,3,0,0,0,0,0,0,0,10242.00
4,0,26,Male,5,3,1,1,0,1,24,...,1,0,0,0,0,0,0,0,200,1489.00


# Pre-processing Pipeline

Indetifying the categorical and the numerical columns

In [6]:
categorical_columns = ['A3', 'A4', 'A5', 'A9', 'A12', 'A16', 'A17', 'A18', 'A19', 'A20'] # Acccording to the dataset description
numerical_columns = [col for col in credit_score.columns if col not in categorical_columns and col != 'A1']

display(categorical_columns)
display(numerical_columns)

['A3', 'A4', 'A5', 'A9', 'A12', 'A16', 'A17', 'A18', 'A19', 'A20']

['A2',
 'A6',
 'A7',
 'A8',
 'A10',
 'A11',
 'A13',
 'A14',
 'A15',
 'A21',
 'A22',
 'A23',
 'A24',
 'A25',
 'A26',
 'A27',
 'A28',
 'A29',
 'A30',
 'A31',
 'A32',
 'A33',
 'A34',
 'A35',
 'A36',
 'A37',
 'A38',
 'A39',
 'A40',
 'A41',
 'A42',
 'A43',
 'A44',
 'A45']

Making the pre-processing pipeline

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [8]:
credit_score_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_columns)
    ]
)

credit_score_preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [9]:
joblib.dump(credit_score_preprocessor, './Data/credit_score_preprocessor.pkl')

['./Data/credit_score_preprocessor.pkl']

# Addressing the missing values in the target feature

Training a random forest classifier to impute the dataset

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [17]:
imputer_pipeline = Pipeline(steps=[
    ('preprocessor', credit_score_preprocessor),
    ('imputer_model', RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=42))
])

In [18]:
rated_data = credit_score[credit_score['A1'] != 0]
unrated_data = credit_score[credit_score['A1'] == 0]

rated_data

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,4,31,Female,5,3,1,3,0,1,24,...,0,4,0,0,0,0,0,0,0,28553.30
1,4,31,Female,5,3,1,3,0,1,24,...,0,2,0,0,0,0,0,0,0,10410.00
2,3,26,Female,7,3,1,0,0,1,24,...,0,7,0,104,0,0,0,0,0,10979.06
5,4,31,Female,7,3,1,1,0,1,24,...,10,3,0,0,0,0,0,0,500,9163.00
6,4,37,Male,4,4,2,0,1,2,24,...,5,0,0,0,0,0,0,0,0,1587.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1852,4,37,Female,5,4,1,0,0,1,12,...,0,4,0,0,0,0,0,0,0,6127.15
1853,3,27,Female,5,3,1,0,0,1,12,...,14,1,0,0,0,0,2,0,0,1956.00
1857,3,36,Female,5,2,2,2,1,2,24,...,31,14,0,28,0,1,3,4,650,43925.00
1858,2,29,Female,5,3,1,0,0,1,12,...,1,6,0,0,0,0,0,3,0,9578.89


In [19]:
imputer_pipeline.fit(
    rated_data.drop(columns=['A1']),
    rated_data['A1']
)

predicted_a1 = imputer_pipeline.predict(unrated_data.drop(columns=['A1']))

C:\Users\cesta\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [20]:
credit_score_imputed = credit_score.copy()
credit_score_imputed.loc[credit_score_imputed['A1'] == 0, 'A1'] = predicted_a1

In [21]:
credit_score_imputed.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,4,31,Female,5,3,1,3,0,1,24,...,0,4,0,0,0,0,0,0,0,28553.30
1,4,31,Female,5,3,1,3,0,1,24,...,0,2,0,0,0,0,0,0,0,10410.00
2,3,26,Female,7,3,1,0,0,1,24,...,0,7,0,104,0,0,0,0,0,10979.06
3,4,63,Male,7,3,3,2,0,2,3,...,0,3,0,0,0,0,0,0,0,10242.00
4,4,26,Male,5,3,1,1,0,1,24,...,1,0,0,0,0,0,0,0,200,1489.00


In [22]:
print("Target A1 Distribution After Imputation:")
display(credit_score_imputed['A1'].value_counts().sort_index())

Target A1 Distribution After Imputation:


A1
1       6
2      32
3     124
4    1690
5      15
Name: count, dtype: int64

Results show that compared to the original and after imputation using random forest, since in the non imputed data the class 4 still domintad the fitting was unbalanced making the predictions biased towards the class 4. We need to find other way of imputation

Imputation using KNN

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

In [24]:
knn_imputer_pipeline = Pipeline(steps=[
    ('preprocessor', credit_score_preprocessor),
    ('imputer_model', KNeighborsClassifier(n_neighbors=3, weights='distance'))
])

X_rated = rated_data.drop(columns=['A1'])
y_rated = rated_data['A1']
X_unrated = unrated_data.drop(columns=['A1'])

In [25]:
knn_imputer_pipeline.fit(X_rated, y_rated)
predicted_a1_knn = knn_imputer_pipeline.predict(X_unrated)

C:\Users\cesta\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [26]:
credit_score_imputed = credit_score.copy()
credit_score_imputed.loc[credit_score_imputed['A1'] == 0, 'A1'] = predicted_a1_knn

print("--- KNN Similarity Imputation Breakdown ---")
display(credit_score_imputed['A1'].value_counts().sort_index())

--- KNN Similarity Imputation Breakdown ---


A1
1       8
2      56
3     260
4    1515
5      28
Name: count, dtype: int64

Using KNN hasn't also been really fructuous since the classes are still imbalanced, this can still lead to some bas when trining the model

Rank-Based Quantile Binning (Continuous Rating Index)

In [50]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline

In [51]:
regressor_pipeline = Pipeline(steps=[
    ('preprocessor', credit_score_preprocessor),
    ('regressor', SVR())
])

In [55]:
rated_data = credit_score[credit_score['A1'] != 0]
unrated_data = credit_score[credit_score['A1'] == 0]

regressor_pipeline.fit(
    rated_data.drop(columns=['A1']),
    rated_data['A1']
)

unrated_scores = regressor_pipeline.predict(unrated_data.drop(columns=['A1']))

C:\Users\cesta\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [54]:
unrated_scores

array([3.51456124, 3.85877432, 3.6298599 , ..., 3.3795493 , 3.3401109 ,
       3.91135076], shape=(1391,))

In [53]:

quantiles = [0, 0.012, 0.077, 0.289, 0.968, 1.0]
predicted_a1_binned = pd.qcut(unrated_scores, q=quantiles, labels=[1, 2, 3, 4, 5])

# Apply back to dataset
credit_score_imputed = credit_score.copy()
credit_score_imputed.loc[credit_score_imputed['A1'] == 0, 'A1'] = predicted_a1_binned

print("--- Quantile-Binned Imputation Breakdown ---")
display(credit_score_imputed['A1'].value_counts().sort_index())

--- Quantile-Binned Imputation Breakdown ---


A1
1      23
2     131
3     386
4    1267
5      60
Name: count, dtype: int64

Using a manual approach, which consists of predicting using a regressor then binning the values to target classes, also didn't work out so let's find other solutions.

Using SMOTE

In [31]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [32]:

X_processed = credit_score_preprocessor.fit_transform(credit_score.drop(columns=['A1']))
y = credit_score['A1']

mask_rated = (y != 0)
X_rated = X_processed[mask_rated]
y_rated = y[mask_rated]
X_unrated = X_processed[~mask_rated]

In [39]:

smote = SMOTE(k_neighbors=5, random_state=42)
X_rated_smote, y_rated_smote = smote.fit_resample(X_rated, y_rated)

display(pd.Series(y_rated_smote).value_counts().sort_index())

A1
1    323
2    323
3    323
4    323
5    323
Name: count, dtype: int64

In [40]:

rf_imputer = RandomForestClassifier(n_estimators=100, random_state=42)
rf_imputer.fit(X_rated_smote, y_rated_smote)
rf_predictions = rf_imputer.predict(X_unrated)

df_rf = credit_score.copy()
df_rf.loc[df_rf['A1'] == 0, 'A1'] = rf_predictions

In [45]:
knn_imputer = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_imputer.fit(X_rated_smote, y_rated_smote)
knn_predictions = knn_imputer.predict(X_unrated)

df_knn = credit_score.copy()
df_knn.loc[df_knn['A1'] == 0, 'A1'] = knn_predictions

In [46]:

imputation_comparison = pd.DataFrame({
    'Original Known (476)': y_rated.value_counts().sort_index(),
    'SMOTE + Random Forest (1,867)': df_rf['A1'].value_counts().sort_index(),
    'SMOTE + KNN (1,867)': df_knn['A1'].value_counts().sort_index()
})

print("\n--- Imputation Comparison ---")
display(imputation_comparison)


--- Imputation Comparison ---


,Original Known (476),"SMOTE + Random Forest (1,867)","SMOTE + KNN (1,867)"
A1,,,
1,6,6,14
2,31,44,165
3,101,181,571
4,323,1616,962
5,15,20,155


After, re-distributing the data using SMOTE, and then predicting the classes using a classifier, we've came to a somewhat usable result using KNN over Random Forest Classifier. This resulted in KNN distributing the classes better than Random Forest so we can take this result.

Storing the dataset

In [47]:
credit_score_imputed = df_knn.copy()

In [48]:
credit_score_imputed.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,A36,A37,A38,A39,A40,A41,A42,A43,A44,A45
0,4,31,Female,5,3,1,3,0,1,24,...,0,4,0,0,0,0,0,0,0,28553.30
1,4,31,Female,5,3,1,3,0,1,24,...,0,2,0,0,0,0,0,0,0,10410.00
2,3,26,Female,7,3,1,0,0,1,24,...,0,7,0,104,0,0,0,0,0,10979.06
3,3,63,Male,7,3,3,2,0,2,3,...,0,3,0,0,0,0,0,0,0,10242.00
4,3,26,Male,5,3,1,1,0,1,24,...,1,0,0,0,0,0,0,0,200,1489.00


In [49]:
credit_score_imputed.to_csv('./Data/credit_score_imputed.csv', index=False)